# Northeast Pacific Marine Heatwaves (GLORYS12): 4-D Tracking of Events as Volumes using marEx

This notebook tracks the marine heatwaves written by [`01_preprocess_extremes.ipynb`](01_preprocess_extremes.ipynb)
through `(time, depth, lat, lon)`, on 36 levels from the surface to 1150 m over 2013-2016. It writes `events_4d.zarr`
and `merges_4d.zarr`, which [`03_visualise_events.ipynb`](03_visualise_events.ipynb) plots. Every parameter is described
under [Tracking in 4-D](https://marex.readthedocs.io/en/latest/guide/tracking.html#tracking-in-4d).

`marEx.tracker` with a `"z"` entry in `dimensions` follows each marine heatwave as a **volume**:

1. **Object identification.** Connected components are labelled in `(depth, lat, lon)` on each day. Levels join
   when their extreme cells are directly above one another, or separated by at most `max_vertical_gap_m` of water.
2. **Morphological pre-processing.** Binary closing and opening with a disk of radius `R_fill` cells (applied level by
   level) fill holes and remove small features, and gaps of up to `T_fill` days are bridged in time.
3. **Volume filter.** Objects are dropped when their volume (m³, from the cell areas and layer thicknesses) lies below
   a quantile of all object volumes. There is no area filter in 4-D.
4. **Temporal overlap, mergers and splits.** Objects on consecutive days are linked by their overlapping volume.
   Mergers and splits are followed, and every merger is logged in `merges_ds`.

The tracked output has `ID_field (time, depth, lat, lon)`, plus per-event `volume` (m³), `footprint_area` (m²), a
volume-weighted horizontal `centroid` and the depths `depth_centroid`, `depth_top` and `depth_bottom` (m, positive down).

Compute: one compute node, with a local cluster of 8 single-threaded workers × 6 GB.

## Setup

In [1]:
from getpass import getuser
from pathlib import Path

import dask
import numpy as np
import pandas as pd
import xarray as xr

import marEx
import marEx.helper as hpc

2026-11-02 16:40:41 - marEx - INFO - [PID:337528] - MarEx logging configured - Level: INFO, Mode: normal


In [2]:
scratch_dir = Path("/scratch") / getuser()[0] / getuser()
out_dir = scratch_dir / "marEx" / "examples" / "4d"  # where 01_preprocess_extremes.ipynb wrote its output
temp_dir = scratch_dir / "marEx" / "examples" / "tmp"  # staging for compute_mode="streaming"

period = slice("2013-01-01", "2016-12-31")  # the period to track, here the whole store
time_chunk = 5  # the tracker's time chunk, a fifth of the store's 25-day chunks

# One compute node: 8 single-threaded workers x 6 GB = 48 GB.
n_workers, threads_per_worker, memory_limit = 8, 1, "6GB"

In [3]:
out_dir.mkdir(parents=True, exist_ok=True)

# graphs of a few hundred MB are expected at this scale
dask.config.set({"distributed.admin.large-graph-warning-threshold": "1GB"})

# memory_limit is set explicitly so the workers stay within the job's allocation (Dask otherwise sizes them from
# the whole node's memory).
client = hpc.start_local_cluster(
    n_workers=n_workers,
    threads_per_worker=threads_per_worker,
    memory_limit=memory_limit,
    scratch_dir=scratch_dir / "marEx" / "dask",
)

# Records wall time, peak memory, spill and worker restarts for each `monitor.stage(...)` block below
monitor = hpc.ResourceMonitor(client)

2026-11-02 16:41:00 - marEx.helper.cluster - INFO - [PID:337528] - Starting local Dask cluster with 8 workers, 1 threads each


2026-11-02 16:41:00 - marEx.helper.dask_config - INFO - [PID:337528] - Configuring Dask for HPC environment


2026-11-02 16:41:00 - marEx.helper.dask_config - INFO - [PID:337528] - Using scratch directory: /scratch/<u>/<user>/marEx/dask


2026-11-02 16:41:00 - marEx.helper.dask_config - INFO - [PID:337528] - Dask temporary directory: /scratch/<u>/<user>/marEx/dask/tmp2b5szctg


2026-11-02 16:41:00 - marEx.helper.dask_config - INFO - [PID:337528] - Dask configuration completed


2026-11-02 16:41:00 - marEx.helper.cluster - INFO - [PID:337528] - System resources: 128 physical cores, 256 logical cores, 251.6GB total memory


... (3 outputs left out)

Hostname: l30739
Forward Port: l30739:8787
Dashboard Link: localhost:8787/status
2026-11-02 16:41:04 - marEx.helper.cluster - INFO - [PID:337528] - Local cluster started successfully - Dashboard: http://127.0.0.1:8787/status


2026-11-02 16:41:04 - marEx.helper.cluster - INFO - [PID:337528] - After cluster startup - Memory Usage - RSS: 571.5MB, Virtual: 25563.1MB, Percent: 0.2%, Available: 245159.7MB


2026-11-02 16:41:04 - marEx.helper.resources - INFO - [PID:337528] - Cluster: 8 workers, 8 threads, memory_limit per worker [8.0] GB (64 GB total)


<env>/lib/python3.10/contextlib.py:142: UserWarning: Creating scratch directories is taking a surprisingly long time. (1.44s) This is often due to running workers on a network file system. Consider specifying a local-directory to point workers to write scratch data to a local disk.
  next(self.gen)
<env>/lib/python3.10/contextlib.py:142: UserWarning: Creating scratch directories is taking a surprisingly long time. (1.45s) This is often due to running workers on a network file system. Consider specifying a local-directory to point workers to write scratch data to a local disk.
  next(self.gen)
<env>/lib/python3.10/contextlib.py:142: UserWarning: Creating scratch directories is taking a surprisingly long time. (1.46s) This is often due to running workers on a network file system. Consider specifying a local-directory to point workers to write scratch data to a local disk.
  next(self.gen)
<env>/lib/python3.10/contextlib.py:142: UserWarning: Creating scratch directories is taking a surpri

## 1. The Detected Extremes and the Geometry

The store contains the boolean `extreme_events`, the wet `mask`, the layer interfaces `depth_bnds` (m) and the horizontal
`cell_area` (m²). The mask is required in 4-D, because a surface mask cannot describe the sea floor. The interfaces give
each level's thickness and the gaps between levels, and the cell volume is `cell_area × thickness`. GLORYS12's static
store gives the thickness per level, not per cell, so it cannot describe partial bottom cells. The standard thickness
is used everywhere, which overstates the volume of cells over the sea floor slightly.

In [4]:
ds = xr.open_zarr(out_dir / "extremes_glorys12_4d.zarr").sel(time=period)

# Space, including depth, stays whole because the labelling and the morphology are global in space. Time is the axis
# to chunk. The store is chunked 25 days deep, and five days keeps the client's merge-loop buffers small.
extreme_events = ds.extreme_events.chunk({"time": time_chunk, "depth": -1, "lat": -1, "lon": -1})
mask = ds.mask.load()
layer_bounds = ds.depth_bnds.load()
cell_area = ds.cell_area.load()
extreme_events

<xarray.DataArray 'extreme_events' (time: 1461, depth: 36, lat: 361, lon: 601)> Size: 11GB
dask.array<rechunk-merge, shape=(1461, 36, 361, 601), dtype=bool, chunksize=(5, 36, 361, 601), chunktype=numpy.ndarray>
Coordinates:
  * depth    (depth) float32 144B 0.494 1.541 2.646 ... 763.3 902.3 1.062e+03
  * lat      (lat) float32 1kB 30.0 30.08 30.17 30.25 ... 59.75 59.83 59.92 60.0
  * lon      (lon) float32 2kB -170.0 -169.9 -169.8 ... -120.2 -120.1 -120.0
  * time     (time) datetime64[ns] 12kB 2013-01-01 2013-01-02 ... 2016-12-31

## 2. Tracking Events as Volumes

The parameters, with the reason for each:

- `R_fill=8`: closing and opening with a disk of 8 cells (about 0.7°, or 74 km north-south) per level removes speckle and fills holes
  smaller than a mesoscale eddy. The gridded examples use 12 cells at 0.25°, a disk of 3°. The same physical size at
  1/12° would be 36 cells, a larger and much more expensive structuring element than this regional example needs.
- `T_fill=2`: bridges gaps of up to two days. The gridded examples use 4 days. A shorter bridge is cheaper, since the closing
  in time is applied to every voxel.
- `max_vertical_gap_m=50`: two layers with extreme cells join across at most 50 m of non-extreme water. Levels range from 1 m thick
  at the surface to 171 m at depth, so a level thicker than 50 m joins its neighbours only when they are adjacent, while
  near the surface a thin non-extreme layer between two hot ones does not split an event.
- `volume_filter_quartile=0.5`: drops the smaller half of the object volumes, the counterpart of the default
  `area_filter_quartile=0.5` in 3-D.
- `overlap_threshold=0.5` (the default): objects are linked when they overlap by at least half of the smaller volume.
- `grid_resolution` is not used, because `cell_areas` contains the true cell areas in m².

The 2013-2016 ID field is about 46 GB as `int32`, so the tracker runs with `compute_mode="streaming"`. The staging
directory outlives `run()`, so the result is written first and the directory then released with `marEx.clear_staging`.
See [Memory Parameters](https://marex.readthedocs.io/en/latest/guide/tracking.html#compute-modes-and-larger-than-memory-tracking).

In [5]:
with monitor.stage("Track: volumes, merging & splitting"):
    tracker = marEx.tracker(
        extreme_events,
        mask,
        dimensions={"time": "time", "z": "depth", "y": "lat", "x": "lon"},  # the "z" entry selects 4-D tracking
        layer_bounds=layer_bounds,  # (depth, 2) interfaces in m
        cell_areas=cell_area,  # m^2
        max_vertical_gap_m=50.0,
        volume_filter_quartile=0.5,  # drop the smallest half of the object volumes
        R_fill=8,
        T_fill=2,
        allow_merging=True,
        regional_mode=True,  # a box, not the globe: no longitude wrap-around
        coordinate_units="degrees",
        compute_mode="streaming",
        temp_dir=str(temp_dir),
    )
    events_ds, merges_ds = tracker.run(return_merges=True)
    events_ds.to_zarr(out_dir / "events_4d.zarr", mode="w")
    merges_ds.to_zarr(out_dir / "merges_4d.zarr", mode="w")

marEx.clear_staging(events_ds)
events_ds = xr.open_zarr(out_dir / "events_4d.zarr", chunks={})
merges_ds = xr.open_zarr(out_dir / "merges_4d.zarr", chunks={})
print(f"{events_ds.sizes['ID']} events, {merges_ds.sizes['merge_ID']} mergers")
events_ds

2026-11-02 16:41:09 - marEx.track.volumetric.tracker4d - INFO - [PID:337528] - Initialising MarEx 4-D tracker


2026-11-02 16:41:09 - marEx.track.volumetric.tracker4d - INFO - [PID:337528] - Parameters: R_fill=8, T_fill=2, volume_filter_quartile=0.5, volume_filter_absolute=None, max_vertical_gap_m=50.0


2026-11-02 16:41:10 - marEx.core.compute_mode - INFO - [PID:337528] - Streaming staging directory: /scratch/<u>/<user>/marEx/examples/tmp/marex_stage_337528_12cbdb18


2026-11-02 16:41:10 - marEx.track.tracker - INFO - [PID:337528] - Starting complete tracking pipeline


2026-11-02 16:41:10 - marEx.track.tracker - INFO - [PID:337528] - Step 1/3: Data preprocessing


2026-11-02 16:41:10 - marEx.track.tracker - INFO - [PID:337528] - Starting Data preprocessing


... (70 outputs left out)

2026-11-02 17:55:01 - marEx.track.tracker - INFO - [PID:337528] - Tracking pipeline completed successfully - 912 events identified


Track: volumes, merging & splitting: 74.7 min, peak worker memory 18.1 GB of 64 GB (largest worker 3.0 of 8.0 GB), client 1.8 GB, spilled 0.0 GB, restarts 0


912 events, 4617 mergers


<xarray.Dataset> Size: 46GB
Dimensions:         (ID: 912, time: 1461, depth: 36, lat: 361, lon: 601,
                     component: 2, sibling_ID: 18)
Coordinates:
  * ID              (ID) int32 4kB 1 2 3 4 5 6 7 ... 906 907 908 909 910 911 912
  * component       (component) int64 16B 0 1
  * depth           (depth) float32 144B 0.494 1.541 2.646 ... 902.3 1.062e+03
  * lat             (lat) float32 1kB 30.0 30.08 30.17 ... 59.83 59.92 60.0
  * lon             (lon) float32 2kB -170.0 -169.9 -169.8 ... -120.1 -120.0
  * time            (time) datetime64[ns] 12kB 2013-01-01 ... 2016-12-31
Dimensions without coordinates: sibling_ID
Data variables: (12/13)
    ID_field        (time, depth, lat, lon) int32 46GB dask.array<chunksize=(1, 36, 361, 601), meta=np.ndarray>
    centroid        (component, time, ID) float64 21MB dask.array<chunksize=(1, 1, 912), meta=np.ndarray>
    depth_bottom    (time, ID) float64 11MB dask.array<chunksize=(1, 912), meta=np.ndarray>
    depth_centroid  (time, ID) float64 11MB dask.array<chunksize=(1, 912), meta=np.ndarray>
    depth_top       (time, ID) float64 11MB dask.array<chunksize=(1, 912), meta=np.ndarray>
    footprint_area  (time, ID) float64 11MB dask.array<chunksize=(1, 912), meta=np.ndarray>
    ...              ...
    mean_thickness  (time, ID) float64 11MB dask.array<chunksize=(1, 912), meta=np.ndarray>
    merge_ledger    (time, ID, sibling_ID) int32 96MB dask.array<chunksize=(1, 912, 18), meta=np.ndarray>
    presence        (time, ID) bool 1MB dask.array<chunksize=(1, 912), meta=np.ndarray>
    time_end        (ID) datetime64[ns] 7kB dask.array<chunksize=(912,), meta=np.ndarray>
    time_start      (ID) datetime64[ns] 7kB dask.array<chunksize=(912,), meta=np.ndarray>
    volume          (time, ID) float64 11MB dask.array<chunksize=(1, 912), meta=np.ndarray>
Attributes: (12/20)
    N_events_final:                912
    N_objects_filtered:            20626
    N_objects_prefiltered:         41251
    R_fill:                        8
    T_fill:                        2
    accepted_volume_fraction:      0.9801262559370476
    ...                            ...
    spatial_connectivity:          8-neighbour horizontal on each level; vert...
    total_merges:                  4617
    vertical_geometry:             bounds
    volume_filter_quartile:        0.5
    volume_threshold (m^3):        2536756914852.992
    volume_units:                  m3

In [6]:
merges_ds

<xarray.Dataset> Size: 1MB
Dimensions:          (merge_ID: 4617, child_idx: 18, parent_idx: 18)
Dimensions without coordinates: merge_ID, child_idx, parent_idx
Data variables:
    child_IDs        (merge_ID, child_idx) int32 332kB dask.array<chunksize=(2309, 18), meta=np.ndarray>
    merge_time       (merge_ID) datetime64[ns] 37kB dask.array<chunksize=(4617,), meta=np.ndarray>
    n_children       (merge_ID) int32 18kB dask.array<chunksize=(4617,), meta=np.ndarray>
    n_parents        (merge_ID) int32 18kB dask.array<chunksize=(4617,), meta=np.ndarray>
    overlap_volumes  (merge_ID, parent_idx) float64 665kB dask.array<chunksize=(2309, 18), meta=np.ndarray>
    parent_IDs       (merge_ID, parent_idx) int32 332kB dask.array<chunksize=(2309, 18), meta=np.ndarray>
Attributes:
    fill_value:  -1

## 3. The Largest Events

Per-event properties are small enough to load. The volume filter kept 20,626 of the 41,251 objects, 98% of their volume,
and these cluster into 912 events with 4,617 logged mergers. The table lists the largest events by peak volume, with their
position on the day of that peak. Four of the ten, and 31% of all events, reach the bottom of the input at 1150 m on that
day, so their volumes are lower bounds. Event 90 is the largest and the longest, at 1334 days with its peak in February
2016. Event 400 peaks in August 2015 and is the main event of the next notebook.

In [7]:
volume = events_ds.volume.load()  # (time, ID), m^3
centroid = events_ds.centroid.load()  # (component, time, ID); component 0 is latitude, 1 is longitude
peak_time = volume.idxmax("time")
at_peak = {"time": peak_time}

table = pd.DataFrame(
    {
        "peak volume (1e15 m3)": volume.max("time").values / 1e15,
        "peak date": peak_time.dt.strftime("%Y-%m-%d").values,
        "lat": centroid.sel(component=0).sel(**at_peak).values,
        "lon": centroid.sel(component=1).sel(**at_peak).values,
        "top (m)": events_ds.depth_top.sel(**at_peak).values,
        "centroid (m)": events_ds.depth_centroid.sel(**at_peak).values,
        "bottom (m)": events_ds.depth_bottom.sel(**at_peak).values,
        "start": events_ds.time_start.dt.strftime("%Y-%m-%d").values,
        "duration (days)": ((events_ds.time_end - events_ds.time_start).dt.days + 1).values,
    },
    index=pd.Index(events_ds.ID.values, name="ID"),
)
table.sort_values("peak volume (1e15 m3)", ascending=False).head(10).round(2)

,peak volume (1e15 m3),peak date,lat,lon,top (m),centroid (m),bottom (m),start,duration (days)
ID,,,,,,,,,
90,0.38,2016-02-09,42.15,-144.82,71.48,441.95,1149.96,2013-04-18,1334
400,0.29,2015-08-10,44.16,-135.24,142.50,758.39,1149.96,2014-10-07,636
436,0.22,2015-12-05,39.98,-140.94,495.13,846.21,1149.96,2014-11-25,561
392,0.16,2015-07-11,46.19,-131.39,0.00,87.83,589.62,2014-09-21,833
2,0.15,2013-01-02,37.84,-155.05,71.48,482.87,1149.96,2013-01-01,143
414,0.15,2016-05-10,56.02,-150.75,0.00,125.25,589.62,2014-10-24,800
437,0.11,2016-02-17,43.47,-134.95,37.23,231.49,589.62,2014-11-26,767
633,0.10,2016-12-06,37.79,-127.04,0.00,156.07,495.13,2015-11-22,406
182,0.09,2014-01-07,50.01,-156.96,0.00,40.23,347.39,2013-09-28,371


## Wall Time and Memory

Peak memory summed over workers, the largest single worker, and the client (this kernel), per stage. The split and merge is
42.7 of the 74.7 minutes. Nothing spilled, no worker restarted and the largest peaked at 3.0 of 8 GB. The outputs here come from
a run at 8 × 8 GB. The 6 GB per worker set above is sized from that 3.0 GB peak and has not itself been run.

In [8]:
monitor.summary()

,wall (min),peak worker memory (GB),peak per worker (GB),peak client memory (GB),peak spilled to disk (GB),worker restarts
stage,,,,,,
"Track: volumes, merging & splitting",74.65,18.15,2.96,1.78,0.0,0


In [9]:
client.close()